# Notebook d'exemple pour ouvrir un .fits du scan du polariseur

Edited by Louise 

In [ ]:
import numpy as np
from astropy.io import fits

# Pour les graphiques
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

%matplotlib inline

# Get the data

In [ ]:
# Define the path to the FITS file and the scan name
data_path = "C:\\Users\\Administrator\\Documents\\Scripts_Commande_VNA\\CosmoCal_data\\"
scan = '20260312_160306'

# Open the FITS file and view its structure
hdul = fits.open(data_path + f"Scan_{scan}.fits")
hdul.info()  # View structure

In [ ]:
# Access the header of the primary HDU (Header Data Unit)
header = hdul[0].header
header

In [ ]:
# Extract relevant parameters from the header
alpha_min = header['ALPH_MI']
alpha_max = header['ALPH_MA']
step = header['STEP']
nfreq = header['POINTS']
traces = ['S21', 'S12', 'S11', 'S22']

# Polariser angle array along the scan
alpha = np.arange(alpha_min, alpha_max + step, step)
print(alpha.shape)

In [ ]:
freq_samples = hdul[0].data *1e-9  # Access frequency samples [GHz]
mag = hdul[1].data  # Access magnitude [dB]
phi = hdul[2].data  # Access phase [deg]

print(mag.shape)  # Check shape of magnitude data [(nstep, nS, points)]
nstep, nS, nfreqs = mag.shape
print(f"Number of steps: {nstep}, number of S-parameters: {nS}, number of frequency points: {nfreqs}")

# Check frequency samples and find the index of the first sample above 127 GHz
fstart_ind = np.argwhere(freq_samples > 127)[0][0]
print(f"First frequency sample above 127 GHz is at index {fstart_ind}, corresponding to {freq_samples[fstart_ind]:.2f} GHz")

### Convert magnitude from dB to linear scale
mag_lin = 10**(mag/10)

# Plot the data

## En fonction de l'angle du polariseur

In [ ]:
### Plot plusieurs fréquences
freqs = [0,2, 3, 10, 15, 20, 30, 40] #indices de fréquences à tracer
#freqs = np.arange(fstart_ind, nfreqs) # Tracer à partir de la première fréquence au-dessus de 127 GHz

fig, ax = plt.subplots(2, 2, figsize=(12, 9))
ax = ax.ravel()

# Set-up the color mapping based on the actual frequency values
freq_values = freq_samples[freqs]
norm = Normalize(vmin=freq_values.min(), vmax=freq_values.max())
cmap = plt.cm.get_cmap('viridis')

for i, f in enumerate(freqs):
    color = cmap(norm(freq_samples[f]))
    ax[0].plot(alpha, mag_lin[:, 0, f], '.-', color=color) # S21
    ax[1].plot(alpha, mag_lin[:, 1, f], '.-', color=color) # S12
    ax[2].plot(alpha, mag_lin[:, 2, f], '.-', color=color) # S11
    ax[3].plot(alpha, mag_lin[:, 3, f], '.-', color=color) # S22

ax[0].set_xlabel("alpha (°)")
ax[0].set_ylabel("S21")
ax[0].grid(10)

ax[1].set_xlabel("alpha (°)")
ax[1].set_ylabel("S12")
ax[1].grid(10)

ax[2].set_xlabel("alpha (°)")
ax[2].set_ylabel("S11")
ax[2].grid(10)

ax[3].set_xlabel("alpha (°)")
ax[3].set_ylabel("S22")
ax[3].grid(10)

# Créer une seule colorbar pour tous les plots
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # nécessaire pour colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Fréquence [GHz]')

fig.tight_layout(rect=[0, 0, 0.9, 1])

## En fonction de la fréquence

In [ ]:
# Indices des angles à tracer (entre 0 et 38)
alp = np.arange(0, 39) 
# alp = [0, 10, 20, 23]

fig, ax = plt.subplots(2, 2, figsize=(10, 10))
ax = ax.ravel()

# Set-up the color mapping based on the actual frequency values
alpha_values = alpha[alp]
norm = Normalize(vmin=alpha_values.min(), vmax=alpha_values.max())
cmap = plt.cm.get_cmap('plasma')

for a in alp:
    color = cmap(norm(alpha[a]))
    ax[0].plot(freq_samples, mag_lin[a, 0, :], '.-', color=color) # S21
    ax[1].plot(freq_samples, mag_lin[a, 1, :], '.-', color=color) # S12
    ax[2].plot(freq_samples, mag_lin[a, 2, :], '.-', color=color) # S11
    ax[3].plot(freq_samples, mag_lin[a, 3, :], '.-', color=color) # S22
ax[0].set_xlabel("Frequency [GHz]")
ax[0].set_ylabel("S21")
#ax[0].set_yscale('log')
ax[0].grid(10)

ax[1].set_xlabel("Frequency [GHz]")
ax[1].set_ylabel("S12")
#ax[1].set_yscale('log')
ax[1].grid(10)

ax[2].set_xlabel("Frequency [GHz]")
ax[2].set_ylabel("S11")
#ax[2].set_yscale('log')
ax[2].grid(10)

ax[3].set_xlabel("Frequency [GHz]")
ax[3].set_ylabel("S22")
#ax[3].set_yscale('log')
ax[3].grid(10)

#créer une colorbar qui montre l'échelle d'indices (ou de valeurs réelles)
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # nécessaire pour colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label(r'Angle $\alpha$ (°)')

fig.tight_layout(rect=[0, 0, 0.9, 1])